2025/04/30 動くか試しに実行
動いた！
５交差検証法を用いた実装。

#References: https://qiita.com/tetsuro731/items/671c81669955d98bb6ff
#概要：LightGBM実行して、テスト結果をMLFlowで確認しよう！

2025/11/16
■運用手順
①Gogle Colabにてmlflowサーバを起動
②起動したURLをmlflow.set_tracking_uri　にセットする
③プログラムを実行する

■変更履歴
1：2026/4/26　モデルを保存するためにpickleを使用
2：(実験中)LightGBM用のOptunaを適用してみる

In [3]:
# Colab環境かどうかを判定（ローカルではスキップ）
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("ローカル環境で実行中（Google Driveはマウントされません）")


Mounted at /content/drive


In [4]:
# Colabの場合のみ、Google Drive内のディレクトリに移動
if IN_COLAB:
    %cd drive/MyDrive/20230614_新機械学習/
else:
    print("ローカル環境: カレントディレクトリを使用")

/content/drive/MyDrive/20230614_新機械学習


In [5]:
!pip install lightgbm
!pip install mlflow

In [ ]:
import seaborn as sns
data = sns.load_dataset('titanic')


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import mlflow
import pickle


# Encode categorical features
le = LabelEncoder()
for col in ['sex', 'embarked', 'who', 'embark_town', 'alive']:
    data[col] = le.fit_transform(data[col])

# 特徴量とターゲットに分割
N_splits = 5
train = data.drop('survived', axis=1)
y = data['survived']

#mlflow.lightgbm.autolog() #mlflow実行用
# 環境に応じてMLflowの追跡先を切り替え
if IN_COLAB:
    mlflow.set_tracking_uri("https://24f1-34-75-156-204.ngrok-free.app")  # Colab用: ngrokのURLをセット
else:
    mlflow.set_tracking_uri("file:./mlruns")  # ローカル用: カレントディレクトリのmlrunsを使用
mlflow.set_experiment("colab-demo")  # demo名かえるならメンテ

kfold = KFold(n_splits=N_splits, shuffle=True, random_state=42)
for fold, (trn_ind, val_ind) in enumerate(kfold.split(train)):
  with mlflow.start_run():
    print(' ')
    print('-'*50)
    print(f'Training fold {fold}/{N_splits}....')
    x_train, x_val = train.iloc[trn_ind], train.iloc[val_ind]
    y_train, y_val = y[trn_ind], y[val_ind]

    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.9,
        'verbosity': -1,
    }
    verbose_eval = 20
    evals_result = {}

    lgb_train = lgb.Dataset(x_train, y_train)
    lgb_valid = lgb.Dataset(x_val, y_val, reference=lgb_train)
    model = lgb.train(
      params = params,
      train_set = lgb_train,
      num_boost_round = 100,
      valid_sets=[lgb_train, lgb_valid],
      callbacks=[lgb.early_stopping(stopping_rounds=20,  verbose=True),
                 lgb.log_evaluation(verbose_eval),
                 lgb.record_evaluation(evals_result)]
    )

    y_pred = model.predict(x_val, num_iteration=model.best_iteration)
    accuracy = accuracy_score(y_val, np.round(y_pred))
    print(f'Fold {fold}, Accuracy: {accuracy}')
    # ※X_trainから1行を抽出
    input_example = pd.DataFrame(x_train.iloc[0]).T  # 最初の1行を入力例として指定

    # モデルをMLflowに保存
    # mlflow.lightgbm.log_model(model, "lightgbm_model", input_example=input_example)

    # パラメータをMLflowに記録
    for param, value in params.items():
        mlflow.log_param(param, value)

    # メトリクスをMLflowに記録
    mlflow.log_metric("best_iteration", model.best_iteration)
    mlflow.log_metric("train_error", evals_result['training']['binary_logloss'][model.best_iteration-1]) # Changed 'train' to 'training' and used best_iteration-1 for index
    mlflow.log_metric("val_error", evals_result['valid_1']['binary_logloss'][model.best_iteration-1]) # Used best_iteration-1 for index
#2026/4/26追加
#モデルを保存するためにpickleを使用
    with open('model.pkl', 'wb') as model_file:
        pickle.dump(model, model_file)

 
--------------------------------------------------
Training fold 0/5....
Training until validation scores don't improve for 20 rounds
[20]	training's binary_logloss: 0.0634963	valid_1's binary_logloss: 0.0647114
[40]	training's binary_logloss: 0.00832195	valid_1's binary_logloss: 0.00847608
[60]	training's binary_logloss: 0.00124187	valid_1's binary_logloss: 0.00127745
[80]	training's binary_logloss: 0.000188665	valid_1's binary_logloss: 0.000196309
[100]	training's binary_logloss: 2.9012e-05	valid_1's binary_logloss: 3.05808e-05
Did not meet early stopping. Best iteration is:
[100]	training's binary_logloss: 2.9012e-05	valid_1's binary_logloss: 3.05808e-05
Fold 0, Accuracy: 1.0


2026/04/25 15:17:50 INFO mlflow.tracking._tracking_service.client: 🏃 View run magnificent-goat-584 at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1/runs/b43a2fb979f34a33a92dc9261123c1cc.
2026/04/25 15:17:50 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1.


 
--------------------------------------------------
Training fold 1/5....
Training until validation scores don't improve for 20 rounds
[20]	training's binary_logloss: 0.0641125	valid_1's binary_logloss: 0.0638302
[40]	training's binary_logloss: 0.0084025	valid_1's binary_logloss: 0.00836669
[60]	training's binary_logloss: 0.00125139	valid_1's binary_logloss: 0.00127699
[80]	training's binary_logloss: 0.000189346	valid_1's binary_logloss: 0.000198416
[100]	training's binary_logloss: 2.90846e-05	valid_1's binary_logloss: 3.13019e-05
Did not meet early stopping. Best iteration is:
[100]	training's binary_logloss: 2.90846e-05	valid_1's binary_logloss: 3.13019e-05
Fold 1, Accuracy: 1.0


2026/04/25 15:17:52 INFO mlflow.tracking._tracking_service.client: 🏃 View run wistful-robin-978 at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1/runs/bf5158053e2b4acba6c6a08c9029eae5.
2026/04/25 15:17:52 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1.


 
--------------------------------------------------
Training fold 2/5....
Training until validation scores don't improve for 20 rounds
[20]	training's binary_logloss: 0.0638393	valid_1's binary_logloss: 0.0642101
[40]	training's binary_logloss: 0.00836679	valid_1's binary_logloss: 0.00841382
[60]	training's binary_logloss: 0.00125598	valid_1's binary_logloss: 0.00124635
[80]	training's binary_logloss: 0.000191618	valid_1's binary_logloss: 0.000187993
[100]	training's binary_logloss: 2.96022e-05	valid_1's binary_logloss: 2.88356e-05
Did not meet early stopping. Best iteration is:
[100]	training's binary_logloss: 2.96022e-05	valid_1's binary_logloss: 2.88356e-05
Fold 2, Accuracy: 1.0


2026/04/25 15:17:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run trusting-kit-58 at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1/runs/79b3e2fe96ce45a48f41212a448823f9.
2026/04/25 15:17:53 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1.


 
--------------------------------------------------
Training fold 3/5....
Training until validation scores don't improve for 20 rounds
[20]	training's binary_logloss: 0.0642013	valid_1's binary_logloss: 0.0637118
[40]	training's binary_logloss: 0.00841411	valid_1's binary_logloss: 0.00835202
[60]	training's binary_logloss: 0.00124797	valid_1's binary_logloss: 0.00126708
[80]	training's binary_logloss: 0.000188298	valid_1's binary_logloss: 0.000196422
[100]	training's binary_logloss: 2.88735e-05	valid_1's binary_logloss: 3.09689e-05
Did not meet early stopping. Best iteration is:
[100]	training's binary_logloss: 2.88735e-05	valid_1's binary_logloss: 3.09689e-05
Fold 3, Accuracy: 1.0


2026/04/25 15:17:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run able-yak-588 at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1/runs/fd7d5349350c4e14a4f368ece54c1751.
2026/04/25 15:17:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1.


 
--------------------------------------------------
Training fold 4/5....
Training until validation scores don't improve for 20 rounds
[20]	training's binary_logloss: 0.0642891	valid_1's binary_logloss: 0.0635975
[40]	training's binary_logloss: 0.00842559	valid_1's binary_logloss: 0.00833786
[60]	training's binary_logloss: 0.00125629	valid_1's binary_logloss: 0.00125856
[80]	training's binary_logloss: 0.00019044	valid_1's binary_logloss: 0.000193086
[100]	training's binary_logloss: 2.92162e-05	valid_1's binary_logloss: 3.00951e-05
Did not meet early stopping. Best iteration is:
[100]	training's binary_logloss: 2.92162e-05	valid_1's binary_logloss: 3.00951e-05
Fold 4, Accuracy: 1.0


2026/04/25 15:17:55 INFO mlflow.tracking._tracking_service.client: 🏃 View run agreeable-turtle-271 at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1/runs/413559bd546841c28f570770b2eba575.
2026/04/25 15:17:55 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://24f1-34-75-156-204.ngrok-free.app/#/experiments/1.


In [15]:
accuracy_score(y_val, np.round(y_pred))

1.0

In [18]:
!ls

 20241113_LightGBM動かない週	     mlopsmlflow_確定版！.ipynb
 20241114_signate試し		     mlrunsample実行.ipynb
 20250101_cursor生成コード実行場所   model.pkl
 20250430_テンプレ		     notebookから.pyに変換
 20250528_Webスクレイピング	     pandas基礎統計分析_20250525.ipynb
 AutoGluon.ipynb		     print文勉強会.ipynb
 ConvertExcel2CSV.ipynb		     README.gdoc
 dataset			     RegressionAnalysis.ipynb
 gemini生成_シンプルML.ipynb	     titanic_datacsv
 Gemini生成のデータ分析.ipynb	     train.csv
'How to mount data to Colab.gdoc'    お試し実行.ipynb
 iris_LGBM_5Fd.ipynb		     サンプル.gsheet
 LightGBM+MLFlow_マスタ.ipynb	     データ基礎分析用.ipynb
'LightGBM+MLFlow_マスタ のコピー'    ボストン住宅価格データ.csv
'LightGBM+MLFlow改造_ver.4 30'	     生成AIの生成AI.ipynb
 matplotlib勉強会.ipynb
